# ML Zoomcamp 2026 — Homework 2

Regression on the 2026 Car Fuel Efficiency dataset.

Submit at: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw02

## Setup

In [2]:
import pandas as pd
import numpy as np

URL = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(URL)

cols = ['engine_displacement', 'horsepower', 'vehicle_weight',
        'model_year', 'fuel_efficiency_mpg']
df = df[cols]

base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
target = 'fuel_efficiency_mpg'

## Q1. Column with missing values

In [3]:
nulls = df.isnull().sum()
print('Q1 nulls per column:')
print(nulls[nulls > 0])

Q1 nulls per column:
horsepower    877
dtype: int64


## Q2. Median horsepower

In [4]:
print('Q2 median horsepower:', df['horsepower'].median())

Q2 median horsepower: 254.0


## Split, model and RMSE helpers

In [5]:
def split(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]
    df_test = df.iloc[idx[n_train + n_val:]]
    return df_train, df_val, df_test

def prepare_X(d, fill_value):
    return d[base].fillna(fill_value).values

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

## Q3. Fill with 0 vs fill with mean

In [6]:
df_train, df_val, df_test = split(df, 42)
y_train = df_train[target].values
y_val = df_val[target].values

# option A: fill with 0
X_train = prepare_X(df_train, 0)
w0, w = train_linear_regression(X_train, y_train)
score_zero = rmse(y_val, w0 + prepare_X(df_val, 0).dot(w))

# option B: fill with TRAIN mean
hp_mean = df_train['horsepower'].mean()
X_train = prepare_X(df_train, hp_mean)
w0, w = train_linear_regression(X_train, y_train)
score_mean = rmse(y_val, w0 + prepare_X(df_val, hp_mean).dot(w))

print('Q3 with 0   :', round(score_zero, 3))
print('Q3 with mean:', round(score_mean, 3))

Q3 with 0   : 2.205
Q3 with mean: 2.202


## Q4. Regularization

In [7]:
print('Q4:')
best = None
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(prepare_X(df_train, 0), y_train, r=r)
    s = round(rmse(y_val, w0 + prepare_X(df_val, 0).dot(w)), 4)
    print(f'   r={r:<6} rmse={s}')
    if best is None or s < best[1]:
        best = (r, s)
print('Q4 best r:', best[0])

Q4:
   r=0      rmse=2.2053
   r=0.01   rmse=2.2058
   r=0.1    rmse=2.2241
   r=1      rmse=2.3492
   r=5      rmse=2.4094
   r=10     rmse=2.4195
   r=100    rmse=2.4292
Q4 best r: 0


## Q5. Seed stability

In [8]:
scores = []
for seed in range(10):
    dtr, dva, dte = split(df, seed)
    ytr, yva = dtr[target].values, dva[target].values
    w0, w = train_linear_regression(prepare_X(dtr, 0), ytr)
    scores.append(rmse(yva, w0 + prepare_X(dva, 0).dot(w)))
print('Q5 std:', round(np.std(scores), 3))

Q5 std: 0.029


## Q6. RMSE on the test set

In [9]:
dtr, dva, dte = split(df, 9)
df_full = pd.concat([dtr, dva])
y_full = df_full[target].values
w0, w = train_linear_regression_reg(prepare_X(df_full, 0), y_full, r=0.001)
score_test = rmse(dte[target].values, w0 + prepare_X(dte, 0).dot(w))
print('Q6 test rmse:', round(score_test, 3))

Q6 test rmse: 2.236
